# Notebook 01 — Pakistani Vehicle Dataset: Inspect Before Training

**Purpose:** Verify the downloaded dataset, inspect make/model labels and images, identify unreadable files and *exact* duplicates, and summarize any publisher train/test split.

**Important:** This notebook does **not** create/overwrite dataset splits, move images, apply augmentation or train a model. Do not use test images to choose preprocessing, hyperparameters or models.

The source specified by the assignment is the **2022 Mendeley Version 1 dataset**, DOI **10.17632/hj3vvx5946.1** (reported: 3,847 images, 48 classes): https://data.mendeley.com/datasets/hj3vvx5946/1

Run all cells from a VS Code notebook kernel using the project's virtual environment. Place the extracted image folders somewhere under `data/pakistan/`. The audit is read-only until its final CSV export.

## 1. Imports and paths

**Intuition:** The image itself is the input; the enclosing class folder supplies the label. A folder called `train` or `test` indicates a predefined split (when present). We must inspect these details before making a new split.

In [ ]:
from collections import Counter
from pathlib import Path
from hashlib import sha256
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import random

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "configs" / "experiments.yaml").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open the notebook from within the GitHub repository (configs/experiments.yaml not found).")

DATA_ROOT = REPO_ROOT / "data" / "pakistan"
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_ROOT}. Download Version 1 from Mendeley, "
        "extract it, and put the extracted folders here."
    )

print("Repository:", REPO_ROOT)
print("Dataset root:", DATA_ROOT)
print("Immediate contents:")
for path in sorted(DATA_ROOT.iterdir(), key=lambda x: x.name.lower()):
    print(" -", path.name, "[folder]" if path.is_dir() else "[file]")

## 2. Inventory images, verify file readability and record labels

We infer the label from the **first folder immediately inside the train/test/validation folder**, or the image's immediate parent if no split folder is present. This is a first-pass assumption: inspect the printed sample paths to confirm it matches the downloaded archive. The notebook does not silently decide which of two overlapping image trees should be used.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SPLIT_NAMES = {
    "train": "train", "training": "train",
    "test": "test", "testing": "test",
    "val": "val", "valid": "val", "validation": "val",
}

def infer_label_and_split(relative_path):
    parts = relative_path.parts
    for idx, folder in enumerate(parts[:-1]):
        canonical = SPLIT_NAMES.get(folder.lower())
        if canonical:
            # Expected layout: .../train/vehicle_class/possibly_nested/image.jpg
            label = parts[idx + 1] if len(parts) > idx + 2 else "(missing class folder)"
            return canonical, label
    return "unspecified", parts[-2] if len(parts) >= 2 else "(missing class folder)"

def file_sha256(path):
    digest = sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

image_paths = sorted(
    (p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS),
    key=lambda p: p.as_posix().lower(),
)
if not image_paths:
    raise RuntimeError("No image files found. Check DATA_ROOT or the downloaded archive.")

records = []
for i, path in enumerate(image_paths):
    rel = path.relative_to(DATA_ROOT)
    split, label = infer_label_and_split(rel)
    row = {
        "relative_path": rel.as_posix(), "split": split,
        "class_name": label, "bytes": path.stat().st_size,
        "width": None, "height": None, "mode": None,
        "sha256": None, "issue": "",
    }
    try:
        with Image.open(path) as im:
            im.verify()  # Detects many invalid/truncated image files
        with Image.open(path) as im:
            im.load()    # Ensures the image can actually be decoded
            row["width"], row["height"], row["mode"] = im.width, im.height, im.mode
        row["sha256"] = file_sha256(path)
    except (OSError, ValueError, SyntaxError) as exc:
        row["issue"] = type(exc).__name__ + ": " + str(exc)[:160]
    records.append(row)

inventory = pd.DataFrame(records)
good = inventory[inventory["issue"].eq("")].copy()
bad = inventory[inventory["issue"].ne("")].copy()
print(f"Files with image extensions: {len(inventory):,}")
print(f"Readable images: {len(good):,}")
print(f"Problematic images: {len(bad):,}")
print(f"Distinct inferred labels: {good['class_name'].nunique()}")
print("Split distribution:", good["split"].value_counts().to_dict())
display(inventory[["relative_path", "split", "class_name", "width", "height", "issue"]].head(12))
if len(bad):
    display(bad[["relative_path", "issue"]].head(20))

## 3. Inspect class balance and split consistency

A model can look accurate while doing badly on rare classes. We therefore count images **per class** and, if present, per split. We must verify that split folders contain the intended class names; the count of **48** is an expectation from the publication, not an assumption to hardcode into the training pipeline.

In [ ]:
counts = (
    good.groupby(["class_name", "split"])
    .size()
    .rename("n_images")
    .reset_index()
)
class_totals = (
    good.groupby("class_name")
    .size()
    .sort_values(ascending=False)
    .rename("n_images")
)
display(counts.head(20))
print("Largest 10 classes:")
display(class_totals.head(10))
print("Smallest 10 classes:")
display(class_totals.tail(10))

coverage = good.groupby("split")["class_name"].nunique()
print("Number of classes in each publisher split:", coverage.to_dict())
train_classes = set(good.loc[good["split"].eq("train"), "class_name"])
test_classes = set(good.loc[good["split"].eq("test"), "class_name"])
print("Classes present in test but absent from train:", sorted(test_classes - train_classes))
print("Classes present in train but absent from test:", sorted(train_classes - test_classes))

ax = class_totals.sort_values().plot.barh(figsize=(10, max(8, len(class_totals) * 0.22)))
ax.set(xlabel="Number of readable images", ylabel="Vehicle class",
       title="Pakistani dataset — observed class distribution")
plt.tight_layout()
plt.show()

splits_found = set(good["split"])
if "unspecified" in splits_found and len(splits_found) > 1:
    print("WARNING: The scan contains BOTH split-specific and unspecified images.")
    print("This may mean the archive contains original and copied/split images.")
    print("Do not train until you choose a single canonical image tree.")
if len(good) != 3847 or good["class_name"].nunique() != 48:
    print("NOTE: Counts differ from the 2022 publication's 3,847 images / 48 classes.")
    print("Inspect nested folders, duplicate trees, invalid files and source version.")

## 4. Check exact duplicates and cross-split leakage

Two images with identical **SHA-256 hashes** have identical file bytes. We report repeated hashes and flag cases where the same file appears in more than one split. **Different JPEG encodings or adjacent video frames may still be visually near-identical and will NOT be found by this check.** Perceptual-hash and visual review is the next audit step.

In [ ]:
repeated = good[good.duplicated("sha256", keep=False)].sort_values("sha256")
cross_split = (
    repeated.groupby("sha256")["split"].nunique()
    if not repeated.empty else pd.Series(dtype=int)
)
cross_split_hashes = cross_split[cross_split > 1].index.tolist()
print("Repeated hash groups:", repeated["sha256"].nunique())
print("Hash groups present in multiple split labels:", len(cross_split_hashes))
if not repeated.empty:
    display(repeated[["relative_path", "split", "class_name", "sha256"]].head(25))
if cross_split_hashes:
    print("WARNING: Exact duplicates cross split boundaries; investigate before training.")

## 5. Look at representative training images

The random seed gives a repeatable sample of one **training** image from each selected class. Check whether class names appear plausible and whether lighting, viewpoints and cropping differ. Test images are not used in this visualization so final test evaluation remains held out.

In [ ]:
train_images = good.loc[good["split"].eq("train")]
if train_images.empty:
    raise RuntimeError("No training images found; check publisher split paths.")
candidates = train_images.sort_values("relative_path").drop_duplicates("class_name")
sample = candidates.sample(n=min(12, len(candidates)), random_state=42)
cols = 4
rows = max(1, (len(sample) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(13, rows * 3.1))
axes = list(axes.flat) if hasattr(axes, "flat") else [axes]
for ax in axes:
    ax.axis("off")
for ax, record in zip(axes, sample.itertuples(index=False)):
    with Image.open(DATA_ROOT / record.relative_path) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(record.class_name[:33], fontsize=9)
plt.tight_layout()
plt.show()

## 6. Export audit metadata (not images)

Store small CSV tables for reproducibility and team review. These are **not** training splits. In the next phase we will decide which supplied train/test structure is canonical and generate a validation manifest without touching the test set.

In [ ]:
OUT = REPO_ROOT / "results" / "data_audit"
OUT.mkdir(parents=True, exist_ok=True)
inventory.to_csv(OUT / "pakistan_inventory.csv", index=False)
counts.to_csv(OUT / "pakistan_class_counts.csv", index=False)
repeated.to_csv(OUT / "pakistan_exact_duplicates.csv", index=False)
print("Audit CSVs written to:", OUT)
print("NEXT: review the observed class/split tree and any cross-split duplicates.")
print("Do not create random splits or start training yet.")

## Reflection questions

1. Why must we retain the publisher's test set rather than randomly reshuffle all available images?
2. If one vehicle class has only 15 samples and another has 250, what could overall accuracy hide?
3. Why can frames from the *same vehicle in the same video* cause train/test leakage even when SHA-256 hashes differ?
4. Why will normalization and resizing matter later for ImageNet-pretrained CNNs?

**Checkpoint for Project HQ:** Record the observed image count, number of classes, split-folder names, unreadable-file count, exact-duplicate groups and screenshot of the class chart. All values are unknown until the notebook is run on your actual downloaded files.